In [6]:
%pip install ollama

  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 1.1 MB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.1 MB 1.1 MB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.1 MB 1.2 MB/s eta 0:00:01
   ----------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import ollama
language_model = "hf.co/bartowski/llama-3.2-1B-instruct-GGUF"
embedding_model = "hf.co/Compendiumi.abs/bge-base-en-v1.5-GGUF"

In [9]:
datasets =[]
with open(r"C:\Users\shiti\Downloads\LPU_GenAI-main\LPU_GenAI-main\cat-facts.txt", "r") as f:
    datasets = []
    with open(r"C:\Users\shiti\Downloads\LPU_GenAI-main\LPU_GenAI-main\cat-facts.txt", "r", encoding="utf-8", errors="replace") as f:
        datasets = f.readlines()
    print(f"Loaded {len(datasets)} datasets from cat-facts.txt")


Loaded 150 datasets from cat-facts.txt


In [10]:
VECTOR_DB = []

Defining the embedding system

In [11]:
def add_chunk_to_vector_db(chunk):
    # Generate embedding for the chunk using the embedding model
    embedding = ollama.embed(model =embedding_model, input =chunk)['embeddings'][0]
    # Store the chunk and its embedding in the vector database
    VECTOR_DB.append((chunk, embedding))

In [12]:
def consine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = sum(a ** 2 for a in vec1) ** 0.5
    magnitude2 = sum(b ** 2 for b in vec2) ** 0.5
    if magnitude1 == 0 or magnitude2 == 0:
        return 0
    return dot_product / (magnitude1 * magnitude2)

In [20]:
def retriever(query, top_k=3):
    if not VECTOR_DB:
        for chunk in datasets:
            if chunk.strip():
                add_chunk_to_vector_db(chunk)
    query_embedding = ollama.embed(model=embedding_model, input=query)['embeddings'][0]
    scores = [(chunk, consine_similarity(query_embedding, emb)) for chunk, emb in VECTOR_DB]
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

results = retriever(query="What is the average lifespan of a cat?")

for chunk, score in results:
    print(f"Score: {score:.4f}, Chunk: {chunk}")

ResponseError: invalid model name (status code: 400)